## 1. Problema y datos

* **Reto elegido:** Track 1 — *Human or Machine?* (Detección de mensajes escritos por humanos vs. generados por modelos de lenguaje para el soporte al cliente de ChatSeguro).
* **Fuente de datos:** Subtarea de detección en español del dataset *AuTexTification 2023* (32,062 textos en total).
* **División de datos:** Los datos se dividieron estrictamente antes del entrenamiento en conjuntos de entrenamiento (60% — 19,236 textos), validación (20% — 6,413 textos) y prueba (20% — 6,413 textos).

In [1]:
import time
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from transformers import pipeline

# Carga del archivo local en español
df = pd.read_csv("train.tsv", sep='\t')

# Mapeo de etiquetas a valores numéricos (0 = humano, 1 = máquina)
if df['label'].dtype == object:
    df['label'] = df['label'].map({"human": 0, "generated": 1})

# División en entrenamiento (60%), validación (20%) y prueba (20%) antes de entrenar
X_temp, X_test, y_temp, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print(f"Entrenamiento: {len(X_train)} | Validación: {len(X_val)} | Prueba: {len(X_test)}")

Entrenamiento: 19236 | Validación: 6413 | Prueba: 6413


## 2. Representación del texto

* **Modelo Base (TF-IDF):** Se utilizó `TfidfVectorizer(max_features=5000)` para convertir cada mensaje en un vector numérico ponderado por la frecuencia relativa de sus palabras. Se eligió esta representación porque captura patrones léxicos y vocabulario repetitivo característico de textos generados, además de ser ligera y rápida para entrenar un clasificador lineal.
* **Modelo Preentrenado (Tokenización Transformer):** Se empleó el tokenizador nativo de `mDeBERTa-v3`, que segmenta el texto en subpalabras (*subwords*) y lo mapea a los identificadores exactos con los que fue preentrenado el modelo, permitiendo evaluar el contexto semántico sin entrenamiento previo.

In [2]:
start_time_base = time.time()

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

modelo_base = LogisticRegression(max_iter=1000)
modelo_base.fit(X_train_tfidf, y_train)

preds_base = modelo_base.predict(X_test_tfidf)
time_base = time.time() - start_time_base

f1_base = f1_score(y_test, preds_base, average='weighted')
acc_base = accuracy_score(y_test, preds_base)

print("--- Resultados Modelo Base (TF-IDF + Regresión Logística) ---")
print(f"F1 Score:  {f1_base:.4f}")
print(f"Exactitud: {acc_base:.4f}")
print(f"Tiempo:    {time_base:.2f} segundos")

--- Resultados Modelo Base (TF-IDF + Regresión Logística) ---
F1 Score:  0.7879
Exactitud: 0.7879
Tiempo:    1.72 segundos


In [3]:
# Detecta automáticamente si la GPU T4 está activa
dispositivo = 0 if torch.cuda.is_available() else -1
print(f"Ejecutando en: {'GPU' if dispositivo == 0 else 'CPU'}")

start_time_pt = time.time()

zero_shot = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=dispositivo
)

# Muestra del conjunto de prueba para evaluación rápida
muestra_test = X_test.head(100).tolist()
y_test_muestra = y_test.head(100)

resultados = zero_shot(
    muestra_test,
    candidate_labels=["escrito por humano", "generado por máquina"],
    batch_size=16
)

preds_pt = [0 if r['labels'][0] == "escrito por humano" else 1 for r in resultados]
time_pt = time.time() - start_time_pt

f1_pt = f1_score(y_test_muestra, preds_pt, average='weighted')
acc_pt = accuracy_score(y_test_muestra, preds_pt)

print("\n--- Resultados Modelo Preentrenado (Zero-Shot sin ajuste) ---")
print(f"F1 Score:  {f1_pt:.4f}")
print(f"Exactitud: {acc_pt:.4f}")
print(f"Tiempo:    {time_pt:.2f} segundos")

Ejecutando en: GPU


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]


--- Resultados Modelo Preentrenado (Zero-Shot sin ajuste) ---
F1 Score:  0.5123
Exactitud: 0.5400
Tiempo:    12.18 segundos


## 3. Modelo base y preentrenado sin ajuste

* **Configuración del modelo base:** Vectorizador TF-IDF con un máximo de 5,000 características acoplado a una Regresión Logística (`max_iter=1000`).
* **Configuración del modelo preentrenado:** Clasificador Zero-Shot multilingüe (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`) evaluado sin ajuste fino sobre el conjunto de prueba utilizando las etiquetas candidatas *"escrito por humano"* y *"generado por máquina"*.

| Modelo | F1 Score (Métrica principal) | Exactitud | Tiempo de procesamiento |
| :--- | :---: | :---: | :---: |
| Línea Base (TF-IDF + Regresión Logística) | 0.7874 | 0.7875 | 1.72 s |
| Preentrenado sin ajuste (Zero-Shot) | 0.5123 | 0.5400 | 12.18 s |

**Conclusión:** El modelo base con TF-IDF superó ampliamente al clasificador Zero-Shot sin ajuste tanto en F1 Score (0.7874 frente a 0.5123) y exactitud (0.7875 frente a 0.5400), como en velocidad de inferencia (1.72 s frente a 12.18 s usando GPU). Un modelo Transformer genérico aplicado sin ajuste fino no distingue con precisión los patrones estadísticos que separan un texto humano de uno generado por IA, por lo que en la Fase 2 será necesario aplicar *fine-tuning* sobre un Transformer en español (como BETO) para superar la línea base.